In [2]:
import os 
import pandas as pd 
from dataclasses import dataclass
from pathlib import Path

In [3]:
pwd

'c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\Research'

In [4]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\src')

In [ ]:
@dataclass
class DataMetricsCalculationConfig:
    root_dir : Path 
    data_path : str 
    local_data_file : str

In [6]:
from ServicenowGamification.Utils.Common import * 
from ServicenowGamification.Constant import * 
from ServicenowGamification.Config import *

In [7]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification')

In [ ]:
class ConfigurationManager:
    def __init__(self,
                 CONFIG_PATH=CONFIG_FILE_PATH,
                 PARAMS_PATH=PARAMS_FILE_PATH
                 ):
         print(CONFIG_PATH)
                 
         print(f"path config yaml {Path(CONFIG_PATH)}")
         self.config=Read_Yaml(Path(CONFIG_PATH))
         self.params=Read_Yaml(Path(PARAMS_PATH))
         self.Metrics_Calculation = self.config.Metrics_Calculation
    def get_metricscalculationconfig(self)->DataMetricsCalculationConfig:
         Config=self.Metrics_Calculation
         create_directories([Config.root_dir])
         data_preprocessingconfig=MetricsCalculationConfig(
              root_dir=Config.root_dir,
              data_path=Config.data_path,
              local_data_file=Config.local_data_file
         )
         return data_preprocessingconfig


In [ ]:
class MetricsCalculation:
    def __init__(self,config:DataMetricsCalculationConfig):
        self.config=config
    def MTTR(self):
        df=pd.read_csv(self.config.data_path)
      
        print(f"\n*-------------- Calculate MTTR(Mean Time To Resolve) ----------------*\n")
        date_columns = ["opened_at", "resolved_at", "closed_at"]
    
        for col in date_columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
        print(df[date_columns].dtypes)    
            #Resolution Time = Resolved Time - Opened Time
            #df["resolution_time"] = (df["resolved_at"] - df["opened_at"])
        df["resolution_time"] = df["resolved_at"] - df["opened_at"]
            # Convert to hours:
            #df["resolution_hours"] = (df["resolution_time"].dt.total_seconds() / 3600)
            #df["resolution_hours"] = (df["resolution_time"].dt.total_seconds() / 3600)
        df["resolution_hours"] = ((df["resolved_at"] - df["opened_at"]).dt.total_seconds() / 3600)
           
        print(f"\n*--------------  overall MTTR (Mean Time To Resolve) ----------------*\n")
           # print(df[["number", "resolution_hours"]].head())
        mttr = df["resolution_hours"].mean()
        print("MTTR:", mttr)
    
        print(df[["number", "resolution_hours"]].head(1))   
        local_path=self.config.local_data_file
        print(f"Preprocessed data saved to: {local_path}")
        df.to_csv(local_path, index=False)     
    
    


    

In [14]:
config = ConfigurationManager()

metrics_calculation_config = config.get_metricscalculationconfig()

metrics_calculation = MetricsCalculation(metrics_calculation_config)   # ✅ Correct
df=metrics_calculation.MTTR()

Config\Config.yaml
path config yaml Config\Config.yaml
Reading YAML : Config\Config.yaml
[2026-08-01 13:53:27,087: INFO: Common: YAML file loaded successfully : Config\Config.yaml]
Reading YAML : params.yaml
[2026-08-01 13:53:27,097: INFO: Common: YAML file loaded successfully : params.yaml]
[2026-08-01 13:53:27,101: INFO: Common: Created directory : Artifacts/MetricsCalculation]

*-------------- Calculate MTTR(Mean Time To Resolve) ----------------*

opened_at      datetime64[us]
resolved_at    datetime64[us]
closed_at      datetime64[us]
dtype: object

*--------------  overall MTTR (Mean Time To Resolve) ----------------*

MTTR: 764.2930265339967
       number  resolution_hours
0  INC0000060         30.388056
Preprocessed data saved to: Artifacts/MetricsCalculation/mttr.csv
